# 08 — Top-rm review and documented overrides

For the production 2025 submission, three rm_ids receive manual overrides. Each is justified below by visible data patterns, **not** by p2024 outcomes.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.data import load_or_build


In [ ]:
ds = load_or_build()
jm = ds.daily.copy(); jm['year'] = jm['date'].dt.year; jm = jm[jm['date'].dt.dayofyear <= 151]
jm_total = jm.groupby(['rm_id','year'])['daily_kg'].sum().unstack().fillna(0)
yearly = ds.daily.copy(); yearly['year'] = yearly['date'].dt.year
yearly_total = yearly.groupby(['rm_id','year'])['daily_kg'].sum().unstack().fillna(0)
for rm in [2130, 3441, 3781]:
    print(f'\nrm_id {rm}:')
    print('  Jan-May totals:', jm_total.loc[rm].to_dict())
    print('  Annual totals:', yearly_total.loc[rm].to_dict())


## Override 1 — rm_id 2130 (continued decline)

5 consecutive years of declining Jan-May totals. Trailing-210d slope-based prediction is dominated by H2-2024 surge and overshoots ~6.1M when the realised 2024 H1 was 3.55M. We override to 0.70 × 2024 H1 cumulative trajectory (≈ 2.5M at May 31).

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
rm = 2130
df = ds.daily[ds.daily['rm_id']==rm].copy()
df['year'] = df['date'].dt.year; df['doy'] = df['date'].dt.dayofyear
df = df[(df['year'] >= 2020) & (df['doy'] <= 151)]
df['cum'] = df.groupby('year')['daily_kg'].cumsum()
for year, g in df.groupby('year'):
    ax.plot(g['doy'], g['cum'], label=str(year))
ax.axhline(0.7 * jm_total.loc[rm, 2024], ls='--', color='red', label='override target (May31)')
ax.set_xlabel('day of year'); ax.set_ylabel('cumulative kg'); ax.set_title(f'rm_id {rm} — Jan-May cumulative by year'); ax.legend(); plt.tight_layout()


## Override 2 — rm_id 3441 (clear silence)

2023 H1 = 3.9M, 2024 H1 = 0. Single-year burst followed by complete silence. Already zero via INTERMITTENT; override documents the decision so a future logic change cannot accidentally re-introduce a positive prediction.

## Override 3 — rm_id 3781 (robust two-year H1, raise α)

2023 H1 = 6.03M, 2024 H1 = 6.53M (1.08× growth). Slope-based prediction collapses (1.68M) because 2024 H2 was lower than H1. Default anchor at α=0.65 lifts to 4.24M; we raise to α=0.80 (5.22M) given the two-year H1 stability. Still leaves a 20% margin against any 2025 decline.